# w9_flash.ipynb — @512 LOSS-FUNCTIONAL LADDER (user top priority)

CE decomposed rung by rung at anchor 512, deployed space only (all structure
/ expander / compressor cells now live in **w9_flash_ablation.ipynb**).
Digit grammar (i units, one dial): the number after i/ai = I weight, same
scale as i2ce's "2"; align == 2x(1-cos) on the sphere, so paper 3:1 == i6
and its align_w-2 rung == i4. References already done: ce@512 m4 0.737, i2ce@512 m4 0.791.
ai2lse was interrupted ~ep550 by a pod stop -- claims go stale after 120 s,
so this pod auto-takes-over and RESUMES from the bucket bundle. AUTO-STOPS
when the ladder is drained.

Ladder, one factor per step:
i2ce -> ai2ce (anchor joins I: rope on top of CE) -> ai2lse (drop CE's
adaptive pull; LSE push stays) -> ai6auni (W&I kernel+budget, both forces
at anchors) -> i6uni (push source -> batch AND no anchor rope = pure W&I);
ai4auni / i4uni = i4 rungs of the two W&I cells (i6 = paper 3:1).


In [ ]:
# constants
import os

REPO = "/workspace/stable-query-latent"
URL = "https://github.com/Nice9Tian/stable-query-latent.git"
DATA_SRC = "/workspace/fusion_cache_w9"
DATA_RAM = "/dev/shm/fusion_cache_w9"
OUT_DIR = "/workspace/w9_out"          # shared campaign out dir

# the loss ladder (arm, cap) -- done cells skip automatically
FLASH = [
    ("wcle_i2ce_icetf", 512),        # I2 + full CE (DONE m4 0.791, skips)
    ("wcle_ai2ce_icetf", 512),       # + anchor joins I (both pulls)
    ("wcle_ai2lse_icetf", 512),      # drop CE's adaptive pull (LSE push stays)
    ("wcle_ai6auni_icetf", 512),   # W&I both-at-anchor (i6 = paper 3:1)
    ("wcle_ai4auni_icetf", 512),   # i4 rung
    ("wcle_i6uni_icetf", 512),     # pure W&I: batch push, no anchor (i6)
    ("wcle_i4uni_icetf", 512),     # i4 rung
]
os.makedirs(OUT_DIR, exist_ok=True)
print(f"{len(FLASH)} ladder cells")


In [ ]:
# FORCE-sync repo to origin/main.
import os, importlib.util
if not os.path.isdir(os.path.join(REPO, ".git")):
    !git clone {URL} {REPO}
%cd {REPO}
!git remote set-url origin {URL}
!git fetch origin main
!git reset --hard origin/main
!git rev-parse --short HEAD
for pkg in ("sklearn", "scipy"):
    if importlib.util.find_spec(pkg) is None:
        !pip -q install scikit-learn scipy
        break
import sys
if REPO not in sys.path:
    sys.path.insert(0, REPO)
sys.path.insert(0, os.path.join(REPO, "Pod"))
import w9_jobs as J
print("machinery loaded")


In [ ]:
# Stage the corpus into RAM (llm views not needed).
import shutil
from pathlib import Path
REQUIRED = ["games.npz", "wiki_eval.npz", "wscan_gal_rev.npz",
            "wscan_pool_rev.npy", "wscan_pool_rev_rid.npy", "wscan_pool_rev_len.npy",
            "ss_queries_rev.npz", "ss_queries_rev_S.npy",
            "wiki_clean_views.npz", "sp_raw_views.npz",
            "tag_labels.npz",
            "wiki_eval_split.json", "_tag_splitM.json"]
src = Path(DATA_SRC)
missing = [f for f in REQUIRED if not (src / f).exists()]
assert not missing, f"missing in {DATA_SRC}: {missing}"
dst = Path(DATA_RAM)
dst.mkdir(parents=True, exist_ok=True)
for f in REQUIRED:
    s, d = src / f, dst / f
    if not d.exists() or d.stat().st_size != s.stat().st_size:
        print(f"staging {f} ({s.stat().st_size/1e9:.2f} GB) ...", flush=True)
        shutil.copyfile(s, d)
DATA_DIR = str(dst)
print("corpus in RAM:", DATA_DIR)


In [ ]:
# Full pool: must be READY on the volume; stage onto fast local storage.
import os, time
from pathlib import Path
from Pod.h5_staging import parallel_copy

ready = Path(DATA_SRC) / "full_pool_READY"
assert ready.exists(), "full pool not READY -- run a campaign notebook's build cell once"
src_v = Path(DATA_SRC) / "full_pool_fp16.npy"
src_m = Path(DATA_SRC) / "full_pool_meta.npz"
need = src_v.stat().st_size + (5 << 30)

def _free(p):
    st = os.statvfs(p)
    return st.f_bavail * st.f_frsize

dest_dir = None
for cand in ("/dev/shm", "/root/data", "/root"):
    Path(cand).mkdir(parents=True, exist_ok=True)
    if _free(cand) > need:
        dest_dir = Path(cand)
        break
if dest_dir is None:
    print("WARNING: no local space -- workers will mmap the NETWORK VOLUME copy.")
    FULL_POOL_PATH = str(src_v)
else:
    dst_v = dest_dir / "full_pool_fp16.npy"
    if dst_v.exists() and dst_v.stat().st_size == src_v.stat().st_size:
        print("local full pool already staged:", dst_v)
    else:
        t0 = time.time()
        tmp = dst_v.with_name(dst_v.name + ".copying")
        print(f"staging {src_v.stat().st_size/2**30:.0f} GiB -> {dst_v} ...", flush=True)
        parallel_copy(src_v, tmp, workers=8)
        os.replace(tmp, dst_v)
        print(f"staged in {(time.time()-t0)/60:.1f} min", flush=True)
    import shutil
    shutil.copyfile(src_m, dest_dir / "full_pool_meta.npz")
    FULL_POOL_PATH = str(dst_v)
print("FULL_POOL_PATH =", FULL_POOL_PATH)


In [ ]:
# Drain the matrix across ALL GPUs (heartbeat claims; done cells skip).
import os, queue, subprocess, threading, time
from pathlib import Path

cdir = Path(OUT_DIR) / "claims"
logd = Path(OUT_DIR) / "logs"
logd.mkdir(parents=True, exist_ok=True)
jobs = queue.Queue()
for arm, cap in FLASH:
    nm = J.fs_label(arm, cap, False, 0, "clean", 16)
    if (Path(OUT_DIR) / J.result_name(nm)).exists():
        print(f"[skip] {nm} done"); continue
    jobs.put((arm, cap, nm))
fails = []

def worker(gpu):
    while True:
        try:
            arm, cap, nm = jobs.get_nowait()
        except queue.Empty:
            return
        if not J.try_claim(cdir, nm):
            print(f"[claim] {nm} held elsewhere -- skipped", flush=True)
            continue
        log = logd / f"{arm}_g{cap}.log"
        cmd = ["python", "-u", J.FS_WORKER,
               "--data-dir", DATA_DIR, "--out-dir", OUT_DIR, "--repo", REPO,
               "--arm", arm, "--anchor-cap", str(cap),
               "--epochs", str(J.FS_EPOCHS), "--ckpt-every", str(J.CKPT_EVERY),
               "--ckpt-seeds", str(J.FS_CKPT_SEEDS),
               "--topup-seeds", str(J.TOPUP_SEEDS),
               "--full-pool", "--full-pool-path", FULL_POOL_PATH,
               "--claim-file", str(cdir / f"{nm}.claim")]
        print(f"[gpu{gpu}] start {nm}", flush=True)
        t0 = time.time()
        with open(log, "w") as fh:
            p = subprocess.run(cmd, stdout=fh, stderr=subprocess.STDOUT,
                               env=dict(os.environ, CUDA_VISIBLE_DEVICES=gpu))
        if p.returncode != 0:
            fails.append((nm, str(log)))
        print(f"[gpu{gpu}] " + ("ok" if p.returncode == 0 else "FAIL")
              + f" {nm} [{(time.time()-t0)/60:.1f} min]", flush=True)

stop_evt = threading.Event()
mon = threading.Thread(target=J._monitor, args=([logd], stop_evt), daemon=True)
mon.start()
gpus = J.detect_gpus()
threads = [threading.Thread(target=worker, args=(g,)) for g in gpus]
t0 = time.time()
for t in threads: t.start()
for t in threads: t.join()
stop_evt.set()
print(f"FLASH drained in {(time.time()-t0)/3600:.1f} h; {len(fails)} failed")
for nm, lg in fails:
    print("  FAILED:", nm, "->", lg)


In [ ]:
# Readout: the loss ladder
import json
import numpy as np
from pathlib import Path
VORD = ["neutral", "noname", "positive", "negative"]
ROWS = [("wcle_ce_cetf", "ce (no-I ref)"),
        ("wcle_i2ce_icetf", "i2ce"),
        ("wcle_ai2ce_icetf", "ai2ce (+anchor rope)"),
        ("wcle_ai2lse_icetf", "ai2lse (-adaptive pull)"),
        ("wcle_ai6auni_icetf", "ai6auni (W&I@anchor, i6)"),
        ("wcle_ai4auni_icetf", "ai4auni (i4)"),
        ("wcle_i6uni_icetf", "i6uni (pure W&I, i6)"),
        ("wcle_i4uni_icetf", "i4uni (i4)"),
        ("wcle_ai6uni_icetf", "ai6uni (mixed src, parked)")]
for arm, lab in ROWS:
    p = Path(OUT_DIR) / f"ft4var_w9_{arm}_fp_best.json"
    if not p.exists():
        print(f"{lab:34s} (missing)"); continue
    d = json.loads(p.read_text())
    runs = d["per_seed"]
    r = {v: np.mean([x[v]["h1"] for x in runs]) for v in VORD}
    print(f"{lab:34s} ep{d.get('best_ep'):>4} "
          + " ".join(f"{v[:3]}:{r[v]:.3f}" for v in VORD)
          + f" m4:{np.mean(list(r.values())):.3f} "
          f"tag:{np.mean([x['noname']['tag'] for x in runs]):.3f}")


In [ ]:
# AUTO-STOP the pod (results are on the network volume).
AUTO_STOP = True
if AUTO_STOP:
    import sys
    if REPO not in sys.path:
        sys.path.insert(0, REPO)
    from VICReg_review import pod_selfstop
    pod_id, api_key, ctl = pod_selfstop.preflight("")
    pod_selfstop.stop_pod(pod_id, api_key, ctl)
else:
    print("AUTO_STOP disabled -- stop the pod yourself.")
